# Part 1.3: data analysis

This notebook presents the churn analysis in execution order so the data source is visible before diagnostics and cleaning.

**Models used**
- Logistic Regression
- Random Forest

**Datasets used**
- `E Commerce Dataset.xlsx`
- `Customer Churn.csv`

## 1. Imports and setup

In [41]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, average_precision_score,
                             roc_auc_score)

SEED = 42
pd.set_option("display.width", 200)

## 2. Feature definitions

In [42]:
# Columns listed explicitly so it is clear what goes where.
EC_NUMERIC = ["Tenure", "CityTier", "WarehouseToHome", "HourSpendOnApp",
              "NumberOfDeviceRegistered", "SatisfactionScore", "NumberOfAddress",
              "Complain", "OrderAmountHikeFromlastYear", "CouponUsed",
              "OrderCount", "DaySinceLastOrder", "CashbackAmount"]

EC_CATEGORICAL = ["PreferredLoginDevice", "PreferredPaymentMode", "Gender",
                  "PreferedOrderCat", "MaritalStatus"]

TEL_NUMERIC = ["Call  Failure", "Complains", "Subscription  Length",
               "Charge  Amount", "Seconds of Use", "Frequency of use",
               "Frequency of SMS", "Distinct Called Numbers", "Tariff Plan",
               "Age", "Customer Value"]

TEL_CATEGORICAL = []          # this dataset is entirely numeric

## 3. Loading the source datasets

The two input files are loaded here, before any diagnostics or cleaning are executed.

In [43]:
# Load source datasets
ecommerce_raw = pd.read_excel("E Commerce Dataset.xlsx", sheet_name="E Comm")
telecom_raw = pd.read_csv("Customer Churn.csv")

print("E-Commerce raw shape:", ecommerce_raw.shape)
print("Telecom raw shape:", telecom_raw.shape)

E-Commerce raw shape: (5630, 20)
Telecom raw shape: (3150, 14)


## 4. Output helper

In [44]:
def title(text):
    print("\n" + "=" * 68)
    print(text)
    print("=" * 68)

## 5. Diagnostic function

In [45]:
# 1. DIAGNOSTICS
# The checks that decided what i did next.
def diagnose(data, name):
    title("DIAGNOSTICS - " + name)
    print("rows:", data.shape[0], " columns:", data.shape[1])

    missing = data.isna().sum()
    missing = missing[missing > 0]
    print("\ncolumns with missing values:", len(missing))
    if len(missing) > 0:
        percent = (missing / len(data) * 100).round(2)
        print(percent.to_string())
        print("-> imputation needed, and it must happen inside the pipeline")

    print("\nexact duplicate rows:", data.duplicated().sum())

    churn_rate = data["Churn"].mean()
    baseline = max(churn_rate, 1 - churn_rate)
    print("\nchurn rate:", round(churn_rate, 4))
    print("majority-class baseline accuracy:", round(baseline, 4))
    print("-> predicting 'no churn' for everyone scores this while finding")
    print("   zero churners, so accuracy cannot be the main metric")
    return baseline

### Running diagnostics on the raw datasets

In [46]:
ecommerce_baseline = diagnose(ecommerce_raw, "E-Commerce")
telecom_baseline = diagnose(telecom_raw, "Telecom")


DIAGNOSTICS - E-Commerce
rows: 5630  columns: 20

columns with missing values: 7
Tenure                         4.69
WarehouseToHome                4.46
HourSpendOnApp                 4.53
OrderAmountHikeFromlastYear    4.71
CouponUsed                     4.55
OrderCount                     4.58
DaySinceLastOrder              5.45
-> imputation needed, and it must happen inside the pipeline

exact duplicate rows: 0

churn rate: 0.1684
majority-class baseline accuracy: 0.8316
-> predicting 'no churn' for everyone scores this while finding
   zero churners, so accuracy cannot be the main metric

DIAGNOSTICS - Telecom
rows: 3150  columns: 14

columns with missing values: 0

exact duplicate rows: 300

churn rate: 0.1571
majority-class baseline accuracy: 0.8429
-> predicting 'no churn' for everyone scores this while finding
   zero churners, so accuracy cannot be the main metric


## 6. Cleaning functions

In [47]:
# 2. Cleaning 

def clean_ecommerce(data):
    title("CLEANING - E-Commerce")

    # The raw diagnostic sees no exact duplicates because CustomerID is unique.
    # Check whether rows match on every predictive field once CustomerID is ignored.
    non_id_duplicates = data.drop(columns=["CustomerID"]).duplicated().sum()
    unique_ids = data["CustomerID"].nunique()

    print(f"Unique CustomerIDs                    : {unique_ids:,} / {len(data):,}")
    print(f"Rows matching on all fields except ID : {non_id_duplicates:,}")
    print("-> matching non-ID rows are kept because their unique CustomerIDs identify")
    print("   them as separate customers rather than duplicate records")

    # CustomerID is an identifier and is removed before modelling.
    data = data.drop(columns=["CustomerID"])
    print("dropped CustomerID")

    # The same value is recorded under two different spellings. If these are
    # not merged, one-hot encoding creates two columns for one concept and
    # splits the signal between them.
    print("\nmerging duplicate category labels:")
    print("  'Phone' -> 'Mobile Phone'")
    print("  'COD' -> 'Cash on Delivery', 'CC' -> 'Credit Card'")
    print("  'Mobile' -> 'Mobile Phone'")
    data["PreferredLoginDevice"] = data["PreferredLoginDevice"].replace(
        "Phone", "Mobile Phone")
    data["PreferredPaymentMode"] = data["PreferredPaymentMode"].replace(
        "COD", "Cash on Delivery")
    data["PreferredPaymentMode"] = data["PreferredPaymentMode"].replace(
        "CC", "Credit Card")
    data["PreferedOrderCat"] = data["PreferedOrderCat"].replace(
        "Mobile", "Mobile Phone")

    # Duplicate rows are kept here. Every CustomerID was unique, so rows that
    # match on everything else are different customers who happen to look alike.
    print("\n-> duplicate rows kept (unique IDs prove distinct customers)")
    return data


def clean_telecom(data):
    title("CLEANING - Telecom")

    # Status records whether the account is active. That is the outcome
    # restated, not something known before churn happens.
    print("leakage check on 'Status':")
    print(pd.crosstab(data["Status"], data["Churn"], normalize="index").round(3))
    print("-> Status restates the outcome, so it is dropped as leakage")

    # Age Group is just Age put into bands, so the two say the same thing.
    print("\n-> Age Group is a banded version of Age, so Age Group is dropped")

    # Duplicates are removed here because there is no customer ID. Identical
    # rows cannot be told apart and could land in both train and test.
    rows_before = len(data)
    data = data.drop_duplicates()
    print("\nduplicates removed:", rows_before - len(data),
          "(", rows_before, "->", len(data), ")")
    print("-> opposite decision to e-commerce, because there is no identifier")

    return data.drop(columns=["Status", "Age Group"])

### Applying cleaning

In [48]:
ecommerce = clean_ecommerce(ecommerce_raw)
telecom = clean_telecom(telecom_raw)
telecom_age_group = telecom_raw.drop_duplicates()["Age Group"]


CLEANING - E-Commerce
Unique CustomerIDs                    : 5,630 / 5,630
Rows matching on all fields except ID : 556
-> matching non-ID rows are kept because their unique CustomerIDs identify
   them as separate customers rather than duplicate records
dropped CustomerID

merging duplicate category labels:
  'Phone' -> 'Mobile Phone'
  'COD' -> 'Cash on Delivery', 'CC' -> 'Credit Card'
  'Mobile' -> 'Mobile Phone'

-> duplicate rows kept (unique IDs prove distinct customers)

CLEANING - Telecom
leakage check on 'Status':
Churn       0      1
Status              
1       0.947  0.053
2       0.527  0.473
-> Status restates the outcome, so it is dropped as leakage

-> Age Group is a banded version of Age, so Age Group is dropped

duplicates removed: 300 ( 3150 -> 2850 )
-> opposite decision to e-commerce, because there is no identifier


## 7. Preprocessing pipeline

In [49]:
# 3. Pipeline
# Imputing and scaling sit inside the pipeline so they are fitted on the
# training data only. Doing them beforehand would leak test information.
def make_preprocessor(numeric_columns, categorical_columns):
    numeric_steps = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ])

    if len(categorical_columns) == 0:
        return ColumnTransformer([("num", numeric_steps, numeric_columns)])

    categorical_steps = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore"))
    ])
    return ColumnTransformer([
        ("num", numeric_steps, numeric_columns),
        ("cat", categorical_steps, categorical_columns)
    ])


def get_feature_names(model):
    """Column names after encoding, with the transformer prefixes removed."""
    raw_names = model.named_steps["prep"].get_feature_names_out()
    clean_names = []
    for name in raw_names:
        clean_names.append(name.replace("num__", "").replace("cat__", ""))
    return clean_names

## 8. Evaluation functions

In [50]:

# 4. Evaluation
def score_model(model, model_name, dataset_name, X_test, y_test, baseline):
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()
    accuracy = accuracy_score(y_test, predictions)

    print("\n" + model_name)
    print("  caught", tp, "churners, missed", fn, ", raised", fp, "false alarms")
    print("  accuracy", round(accuracy, 3), "vs baseline", round(baseline, 3),
          "  <- BELOW baseline" if accuracy < baseline else "")

    return {
        "Dataset": dataset_name,
        "Model": model_name,
        "Precision": round(precision_score(y_test, predictions), 3),
        "Recall": round(recall_score(y_test, predictions), 3),
        "F1": round(f1_score(y_test, predictions), 3),
        "PR-AUC": round(average_precision_score(y_test, probabilities), 3),
        "ROC-AUC": round(roc_auc_score(y_test, probabilities), 3)
    }

## 9. Interpretation and comparison functions

In [51]:

# 5. Insights

def show_odds_ratios(model, dataset_name, top=6):
    """
    Logistic regression coefficients converted to odds ratios.
    exp(coefficient) turns the log-odds the model works in back into a
    multiplier: above 1 raises churn, below 1 lowers it. Features are
    scaled, so read each one per standard deviation.
    """
    print("\nLogistic regression - largest effects (" + dataset_name + "):")
    coefficients = model.named_steps["clf"].coef_[0]
    table = pd.DataFrame({
        "feature": get_feature_names(model),
        "coefficient": coefficients.round(3),
        "odds_ratio": np.exp(coefficients).round(2)
    })
    table["size"] = table["coefficient"].abs()
    table = table.sort_values("size", ascending=False).drop(columns="size")
    print(table.head(top).to_string(index=False))


def show_importances(model, dataset_name, top=6):
    """Random forest importance. Shows association, not causation."""
    print("\nRandom forest - most important features (" + dataset_name + "):")
    table = pd.DataFrame({
        "feature": get_feature_names(model),
        "importance": model.named_steps["clf"].feature_importances_.round(3)
    })
    table = table.sort_values("importance", ascending=False)
    print(table.head(top).to_string(index=False))


def show_missed_customers(model, X_test, y_test, columns, dataset_name):
    """Compares churners the model missed against churners it caught."""
    title("WHO THE MODEL MISSES - " + dataset_name)

    checked = X_test.copy()
    checked["actual"] = y_test.values
    checked["predicted"] = model.predict(X_test)

    missed = checked[(checked["actual"] == 1) & (checked["predicted"] == 0)]
    caught = checked[(checked["actual"] == 1) & (checked["predicted"] == 1)]

    comparison = pd.DataFrame({
        "missed": missed[columns].mean().round(2),
        "caught": caught[columns].mean().round(2)
    })
    print(len(missed), "missed vs", len(caught), "caught")
    print(comparison.to_string())


def show_recall_by_group(model, X_test, y_test, group_column, dataset_name):
    """Recall inside each subgroup, for the diversity and inclusion argument."""
    title("FAIRNESS CHECK - " + dataset_name)

    checked = X_test.copy()
    checked["actual"] = y_test.values
    checked["predicted"] = model.predict(X_test)

    rows = []
    for group in sorted(checked[group_column].dropna().unique()):
        churners = checked[(checked[group_column] == group) &
                           (checked["actual"] == 1)]
        if len(churners) > 0:
            recall = (churners["predicted"] == 1).mean()
            rows.append({group_column: group,
                         "churners": len(churners),
                         "recall": round(recall, 3)})

    print(pd.DataFrame(rows).to_string(index=False))
    print("-> check the churner counts before reading any gap as bias")


def compare_datasets(ecommerce, telecom):
    """The comparison behind the complementary or contradictory answer."""
    title("CROSS-DATASET COMPARISON")

    ec_complaint = ecommerce.groupby("Complain")["Churn"].mean().round(3)
    tel_complaint = telecom.groupby("Complains")["Churn"].mean().round(3)

    print("complaint effect, e-commerce:", ec_complaint[0], "->", ec_complaint[1],
          "(", round(ec_complaint[1] / ec_complaint[0], 1), "x )")
    print("complaint effect, telecom:   ", tel_complaint[0], "->", tel_complaint[1],
          "(", round(tel_complaint[1] / tel_complaint[0], 1), "x )")
    print("-> COMPLEMENTARY in direction, DIFFERENT in magnitude")

    ec_quartiles = ecommerce.groupby(
        pd.qcut(ecommerce["Tenure"], 4), observed=True)["Churn"].mean().round(3)
    tel_quartiles = telecom.groupby(
        pd.qcut(telecom["Subscription  Length"], 4), observed=True)["Churn"].mean().round(3)

    print("\ntenure quartiles, e-commerce:", ec_quartiles.values.tolist())
    print("tenure quartiles, telecom:   ", tel_quartiles.values.tolist())
    print("-> CONTRADICTORY: falls steadily in retail, not in telecom")

    return ec_complaint, tel_complaint, ec_quartiles, tel_quartiles

## 10. Visualisation functions

In [52]:
# 6. Figures
def plot_performance(results):
    positions = np.arange(len(results))
    labels = results["Dataset"] + "\n" + results["Model"]

    plt.figure(figsize=(9, 4.5))
    plt.bar(positions - 0.27, results["Precision"], 0.27, label="Precision")
    plt.bar(positions, results["Recall"], 0.27, label="Recall")
    plt.bar(positions + 0.27, results["F1"], 0.27, label="F1")
    plt.xticks(positions, labels, fontsize=8)
    plt.ylabel("Score")
    plt.ylim(0, 1.05)
    plt.title("Model performance by dataset")
    plt.legend()
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("figure1_model_comparison.png", dpi=200)
    plt.close()
    print("saved figure1_model_comparison.png")


def plot_comparison(ec_complaint, tel_complaint, ec_quartiles, tel_quartiles):
    plt.figure(figsize=(11, 4))

    plt.subplot(1, 2, 1)
    positions = np.arange(2)
    plt.bar(positions - 0.18, [ec_complaint[0], ec_complaint[1]], 0.35,
            label="E-commerce")
    plt.bar(positions + 0.18, [tel_complaint[0], tel_complaint[1]], 0.35,
            label="Telecom")
    plt.xticks(positions, ["No complaint", "Complaint"])
    plt.ylabel("Churn rate")
    plt.title("Complaints: same direction, different size")
    plt.legend()
    plt.grid(axis="y", alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(range(1, 5), ec_quartiles.values, marker="o", label="E-commerce tenure")
    plt.plot(range(1, 5), tel_quartiles.values, marker="s", label="Telecom subscription")
    plt.xticks(range(1, 5))
    plt.xlabel("Quartile, shortest to longest")
    plt.ylabel("Churn rate")
    plt.title("Tenure: falls steadily in one, not the other")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("figure2_cross_dataset.png", dpi=200)
    plt.close()
    print("saved figure2_cross_dataset.png")

## 11. Train, evaluate, interpret, and generate figures

In [53]:
all_results = []

# E-Commerce
title("EVALUATION - E-Commerce")

X = ecommerce.drop(columns=["Churn"])
y = ecommerce["Churn"]
ec_X_train, ec_X_test, ec_y_train, ec_y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

ec_preprocessor = make_preprocessor(EC_NUMERIC, EC_CATEGORICAL)

# class_weight="balanced" makes errors on the rare churn class count more.
# Without it a model can score well by rarely predicting churn at all.
ec_logistic = Pipeline([
    ("prep", ec_preprocessor),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                               random_state=SEED))
])
ec_forest = Pipeline([
    ("prep", ec_preprocessor),
    ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                   min_samples_leaf=2, random_state=SEED,
                                   n_jobs=-1))
])

ec_logistic.fit(ec_X_train, ec_y_train)
ec_forest.fit(ec_X_train, ec_y_train)

all_results.append(score_model(ec_logistic, "Logistic Regression", "E-Commerce",
                               ec_X_test, ec_y_test, ecommerce_baseline))
all_results.append(score_model(ec_forest, "Random Forest", "E-Commerce",
                               ec_X_test, ec_y_test, ecommerce_baseline))

#  Telecom
title("EVALUATION - Telecom")

X = telecom.drop(columns=["Churn"])
y = telecom["Churn"]
tel_X_train, tel_X_test, tel_y_train, tel_y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

tel_preprocessor = make_preprocessor(TEL_NUMERIC, TEL_CATEGORICAL)

tel_logistic = Pipeline([
    ("prep", tel_preprocessor),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                               random_state=SEED))
])
tel_forest = Pipeline([
    ("prep", tel_preprocessor),
    ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                   min_samples_leaf=2, random_state=SEED,
                                   n_jobs=-1))
])

tel_logistic.fit(tel_X_train, tel_y_train)
tel_forest.fit(tel_X_train, tel_y_train)

all_results.append(score_model(tel_logistic, "Logistic Regression", "Telecom",
                               tel_X_test, tel_y_test, telecom_baseline))
all_results.append(score_model(tel_forest, "Random Forest", "Telecom",
                               tel_X_test, tel_y_test, telecom_baseline))

# results
results = pd.DataFrame(all_results)
title("MODEL COMPARISON")

# Display a clean notebook table while keeping the underlying values unchanged.
# Values are generated directly from the model outputs, not typed manually.
from IPython.display import display

styled_results = (
    results.style
        .hide(axis="index")
        .format({
            "Precision": "{:.3f}",
            "Recall": "{:.3f}",
            "F1": "{:.3f}",
            "PR-AUC": "{:.3f}",
            "ROC-AUC": "{:.3f}"
        })
        .set_properties(**{
            "text-align": "left",
            "padding": "10px 16px",
            "font-size": "14px"
        })
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                    ("font-size", "15px"),
                    ("font-weight", "bold"),
                    ("padding", "10px 16px"),
                    ("border-bottom", "1px solid #999")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("border-bottom", "1px solid #ccc")
                ]
            }
        ])
)

display(styled_results)

print("\n-> random forest wins on both, and the gap is mainly precision rather than recall")
print("-> compare PR-AUC with ROC-AUC: ROC-AUC can look optimistic under class imbalance")

# Keep a machine-readable copy of the exact model results.
results.to_csv("model_comparison.csv", index=False)

# Insights
title("CHURN DRIVERS - E-Commerce")
show_odds_ratios(ec_logistic, "E-Commerce")
show_importances(ec_forest, "E-Commerce")

title("CHURN DRIVERS - Telecom")
show_odds_ratios(tel_logistic, "Telecom")
show_importances(tel_forest, "Telecom")

show_missed_customers(ec_forest, ec_X_test, ec_y_test,
                      ["Tenure", "OrderCount", "CashbackAmount"], "E-Commerce")

show_recall_by_group(ec_forest, ec_X_test, ec_y_test, "Gender", "E-Commerce")

telecom_test_with_group = tel_X_test.copy()
telecom_test_with_group["Age Group"] = telecom_age_group.loc[tel_X_test.index]
show_recall_by_group(tel_forest, telecom_test_with_group, tel_y_test,
                     "Age Group", "Telecom")

ec_complaint, tel_complaint, ec_quartiles, tel_quartiles = compare_datasets(
    ecommerce, telecom)

# Figures
title("FIGURES")
plot_performance(results)
plot_comparison(ec_complaint, tel_complaint, ec_quartiles, tel_quartiles)


EVALUATION - E-Commerce

Logistic Regression
  caught 161 churners, missed 29 , raised 205 false alarms
  accuracy 0.792 vs baseline 0.832   <- BELOW baseline

Random Forest
  caught 169 churners, missed 21 , raised 15 false alarms
  accuracy 0.968 vs baseline 0.832 

EVALUATION - Telecom

Logistic Regression
  caught 78 churners, missed 11 , raised 99 false alarms
  accuracy 0.807 vs baseline 0.843   <- BELOW baseline

Random Forest
  caught 76 churners, missed 13 , raised 10 false alarms
  accuracy 0.96 vs baseline 0.843 

MODEL COMPARISON


Dataset,Model,Precision,Recall,F1,PR-AUC,ROC-AUC
E-Commerce,Logistic Regression,0.440,0.847,0.579,0.679,0.885
E-Commerce,Random Forest,0.918,0.889,0.904,0.977,0.995
Telecom,Logistic Regression,0.441,0.876,0.586,0.735,0.914
Telecom,Random Forest,0.884,0.854,0.869,0.891,0.979



-> random forest wins on both, and the gap is mainly precision rather than recall
-> compare PR-AUC with ROC-AUC: ROC-AUC can look optimistic under class imbalance

CHURN DRIVERS - E-Commerce

Logistic regression - largest effects (E-Commerce):
                            feature  coefficient  odds_ratio
            PreferedOrderCat_Others        2.282        9.79
PreferedOrderCat_Laptop & Accessory       -1.876        0.15
                             Tenure       -1.521        0.22
                     CashbackAmount       -0.977        0.38
      PreferedOrderCat_Mobile Phone       -0.967        0.38
                           Complain        0.721        2.06

Random forest - most important features (E-Commerce):
          feature  importance
           Tenure       0.258
   CashbackAmount       0.090
         Complain       0.067
  WarehouseToHome       0.061
DaySinceLastOrder       0.057
  NumberOfAddress       0.052

CHURN DRIVERS - Telecom

Logistic regression - largest effect

## 12. REFERENCES 

#Harris, C. R., Millman, K. J., van der Walt, S. J., Gommers, R., Virtanen, P., Cournapeau, D., Wieser, E., Taylor, J., Berg, S., Smith, N. J., Kern, R., Picus, M., Hoyer, S., van Kerkwijk, M. H., Brett, M., Haldane, A., del Río, J. F., Wiebe, M., Peterson, P., … Oliphant, T. E. (2020). Array programming with NumPy. Nature, 585(7825), 357–362. https://doi.org/10.1038/s41586-020-2649-2
#RMIT University. (2026). Practical Data Science Jupyter notebook template [Assignment 2 Data modelling and presentation by Haswanth]. Course material.
#Scikit Learn. (2025). 3.3. Metrics and scoring: Quantifying the quality of predictions — Scikit-learn 0.22.1 documentation. In scikit-learn.org. https://scikit-learn.org/stable/modules/model_evaluation.html
#scikit-learn. (2014). LogisticRegression. In Scikit-learn.org. https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
#scikit-learn. (2018). Sklearn.Model_selection.Train_test_split — Scikit-learn 0.20.3 documentation. In Scikit-learn.org. https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
#Scikit-Learn. (2025). Sklearn.Ensemble.RandomForestClassifier — Scikit-learn 0.20.3 documentation. In Scikit-learn.org. https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html